# CrewAI Coding Agents and CLI [Production - Module 03]

> **MLCourse - Agentic AI - CrewAI Production**

CrewAI supports coding agents that can write and execute Python code safely
within a sandboxed environment. This module covers the `CodeInterpreterTool`,
the `AGENTS.md` file format, the `crewai create crew` CLI for scaffolding
projects, and how coding agents execute code safely.

## What you will learn

1. How `CodeInterpreterTool` enables agents to write and run Python code.
2. The `AGENTS.md` file format and its role in coding agent configuration.
3. The `crewai create crew` CLI for scaffolding new projects.
4. The `crewai run` command for executing crews.
5. How coding agents write, execute, and validate Python code safely.

## Key takeaways

- `CodeInterpreterTool` provides a sandboxed Python execution environment.
- `AGENTS.md` tells coding agents about project structure and conventions.
- `crewai create crew` generates a complete project skeleton.
- Code execution is wrapped in try/except with output capture.
- Agents can inspect, modify, and rerun code iteratively.

In [ ]:
# ---- Setup: imports, environment, track discovery ---------------------------

import os
import sys
import json
import time
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
load_dotenv(TRACK / ".env", override=False)

api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] OPENAI_API_KEY found -- optional cloud calls will work")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

In [ ]:
# ---- Check Ollama availability --------------------------------------------

OLLAMA_OK = False
try:
    from langchain_ollama import ChatOllama
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    OLLAMA_OK = True
    print("Ollama: ONLINE (llama3.1:8b)")
except Exception as e:
    print("Ollama: OFFLINE --", e)

In [ ]:
# ---- CrewAI imports ---------------------------------------------------------

CREWAI_OK = False
try:
    from crewai import Agent, Task, Crew, Process, LLM
    CREWAI_OK = True
    print("CrewAI version:", __import__("crewai").__version__)
except ImportError as e:
    print("CrewAI not installed:", e)

In [ ]:
# ---- Try importing CodeInterpreterTool -------------------------------------

CODE_TOOL_OK = False
try:
    from crewai_tools import CodeInterpreterTool
    CODE_TOOL_OK = True
    print("CodeInterpreterTool available from crewai_tools")
except ImportError:
    try:
        from crewai.tools import CodeInterpreterTool
        CODE_TOOL_OK = True
        print("CodeInterpreterTool available from crewai.tools")
    except ImportError:
        print("CodeInterpreterTool not available")
        print("Install with: pip install crewai-tools")

## 1. The `AGENTS.md` File Format

When using coding agents, CrewAI looks for an `AGENTS.md` file in the
project root. This file provides context about the project structure,
coding conventions, and tool availability. It is similar to a `README.md`
but specifically targeting AI agents.

```markdown
# Project: My Application

## Structure
- src/ : Source code directory
- tests/ : Test files
- data/ : Data files

## Conventions
- Use type hints in all functions
- Follow PEP 8 style
- Write docstrings for public functions

## Available Tools
- CodeInterpreterTool: Run Python code
- FileReadTool: Read files
- FileWriteTool: Write files
```

The `AGENTS.md` helps coding agents understand what they can and cannot do.

In [ ]:
# ---- Create a sample AGENTS.md file ----------------------------------------

agents_md_content = """# AGENTS.md -- Project Configuration for Coding Agents

## Project Overview
This is a demo project showing how coding agents interact with CrewAI.

## Directory Structure
- src/ : Application source code
- tests/ : Unit and integration tests
- data/ : Training and input data files
- notebooks/ : Jupyter notebooks for exploration

## Coding Conventions
- Python 3.11+ with type hints
- PEP 8 style enforced via ruff
- All public functions must have docstrings
- Use dataclasses for data models
- No wildcard imports (import x, not from x import *)

## Available Tools
- CodeInterpreterTool: Execute Python code in a sandboxed environment
- FileReadTool: Read file contents
- FileWriteTool: Write content to files
- SerperDevTool: Web search (requires SERPER_API_KEY)

## Safety Rules
- Never execute code that modifies system files
- Always validate inputs before processing
- Use try/except for all external calls
- Log all tool invocations for audit trail
"""

# Write the sample AGENTS.md.
demo_dir = TRACK / "03_agentic_ai" / "04_crewai" / "data"
demo_dir.mkdir(parents=True, exist_ok=True)
agents_md_path = demo_dir / "AGENTS.md"

with open(agents_md_path, "w", encoding="utf-8") as f:
    f.write(agents_md_content)

print(f"AGENTS.md written to: {agents_md_path}")
print(f"Content length: {len(agents_md_content)} chars")
print(f"Lines: {agents_md_content.count(chr(10))}")

## 2. The `crewai create crew` CLI

CrewAI provides a CLI command to scaffold a new crew project. This creates
the directory structure, configuration files, and starter code.

```bash
# Create a new crew project
crewai create crew my_project

# This generates:
# my_project/
#   src/my_project/
#     __init__.py
#     main.py        -- Crew definition
#     agents.py      -- Agent definitions
#     tasks.py       -- Task definitions
#     tools.py       -- Custom tool definitions
#   tests/
#     test_main.py   -- Basic tests
#   pyproject.toml   -- Project configuration
#   .env.example     -- Environment template
#   README.md        -- Project documentation
```

In [ ]:
# ---- Show crewai CLI help --------------------------------------------------

import subprocess
try:
    result = subprocess.run(
        [sys.executable, "-m", "crewai", "--help"],
        capture_output=True, text=True, timeout=30
    )
    print("=== crewai CLI help ===")
    print(result.stdout[:2000] if result.stdout else "No output")
except Exception as e:
    print(f"CLI help not available: {e}")
    print("The crewai CLI requires crewai to be installed.")
    print("Install with: pip install crewai")

## 3. Project Structure from `crewai create crew`

The scaffolded project has a standard layout that coding agents understand.
Each file has a specific role:

- `main.py`: Defines the `Crew` class with `@agent` and `@task` decorators.
- `agents.py`: Defines individual agent roles, goals, and backstories.
- `tasks.py`: Defines task descriptions, expected outputs, and agent assignments.
- `tools.py`: Custom tools that agents can use during execution.
- `tests/`: Test files that validate crew behavior.

In [ ]:
# ---- Show scaffolded project structure (simulated) -------------------------

print("=== CrewAI Project Structure ===\n")
project_tree = """
my_project/
|-- src/my_project/
|   |-- __init__.py
|   |-- main.py          # Crew class with @agent, @task decorators
|   |-- agents.py        # Agent definitions (role, goal, backstory)
|   |-- tasks.py         # Task definitions (description, expected_output)
|   |-- tools.py         # Custom tool implementations
|-- tests/
|   |-- test_main.py     # Crew behavior tests
|-- pyproject.toml       # Dependencies and project metadata
|-- .env.example         # Environment variable template
|-- AGENTS.md            # Coding agent instructions
|-- README.md            # Human-readable documentation
"""
print(project_tree)

# Show what main.py looks like in a scaffolded project.
print("=== Sample main.py ===\n")
main_py = '''
from crewai import Agent, Task, Crew, Process
from crewai.project import CrewBase, agent, task, crew

@CrewBase
class MyProject:
    """Main crew definition for MyProject."""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config["researcher"],
            verbose=True,
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config["research_task"],
        )

    @crew
    def crew(self) -> Crew:
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
        )
'''
print(main_py)

## 4. The `crewai run` Command

After scaffolding, you run the crew with:

```bash
# Run the crew with default inputs
crewai run

# Run with specific inputs
crewai run --inputs '{"topic": "AI safety"}'

# Run in debug mode
crewai run --debug
```

The `run` command:
1. Loads the crew from `src/main.py`.
2. Validates agent and task configurations.
3. Executes the crew with the specified inputs.
4. Outputs results to stdout and optionally to a file.

In [ ]:
# ---- Show crewai run help --------------------------------------------------

try:
    result = subprocess.run(
        [sys.executable, "-m", "crewai", "run", "--help"],
        capture_output=True, text=True, timeout=30
    )
    print("=== crewai run --help ===")
    print(result.stdout[:2000] if result.stdout else "No output")
except Exception as e:
    print(f"CLI help not available: {e}")

## 5. CodeInterpreterTool -- Sandboxed Code Execution

The `CodeInterpreterTool` lets agents write and execute Python code in a
controlled environment. The tool:
1. Receives code from the agent.
2. Executes it in a subprocess with timeout.
3. Captures stdout, stderr, and return code.
4. Returns the output to the agent for further processing.

This is essential for coding agents that need to perform calculations,
generate visualizations, or process data during task execution.

In [ ]:
# ---- Demonstrate CodeInterpreterTool usage --------------------------------

if CODE_TOOL_OK:
    # Create the tool instance.
    code_tool = CodeInterpreterTool()
    print("CodeInterpreterTool created")
    print(f"Tool name: {code_tool.name}")
    print(f"Tool description: {code_tool.description[:100]}...")
    print()

    # Show the tool's configuration options.
    print("Configuration options:")
    print("  - code: The Python code to execute")
    print("  - timeout: Maximum execution time (default: 30s)")
    print("  - libraries: Allowed imports (default: all standard lib)")
else:
    print("[SKIP] CodeInterpreterTool not available")
    print("Showing reference pattern instead.\n")

# Reference pattern for safe code execution.
print("=== Safe Code Execution Pattern ===\n")
safe_code_pattern = '''
import subprocess
import sys
import tempfile
import os

def safe_execute(code: str, timeout: int = 30) -> dict:
    """Execute Python code in a sandboxed subprocess.

    Args:
        code: Python source code to execute.
        timeout: Maximum seconds before killing the process.

    Returns:
        Dictionary with stdout, stderr, returncode, and success flag.
    """
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False
    ) as f:
        f.write(code)
        tmp_path = f.name

    try:
        result = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout,
            env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
        )
        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode,
            "success": result.returncode == 0,
        }
    except subprocess.TimeoutExpired:
        return {
            "stdout": "",
            "stderr": f"Execution timed out after {timeout}s",
            "returncode": -1,
            "success": False,
        }
    finally:
        os.unlink(tmp_path)
'''
print(safe_code_pattern)

## 6. Agent Writes and Runs Python Code (Demo)

We demonstrate a coding agent that writes a Python calculation, executes
it, and uses the result. The agent uses `CodeInterpreterTool` to run
the code and verify the output.

In [ ]:
# ---- Demo: agent writes and runs a calculation -----------------------------

if CREWAI_OK and OLLAMA_OK:
    ollama_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

    # Create a coding agent with the CodeInterpreterTool.
    coding_agent = Agent(
        role="Python Developer",
        goal="Write and execute Python code to solve computational problems.",
        backstory=(
            "You are an expert Python developer. You write clean, efficient "
            "code and verify your solutions by running them. You always "
            "include error handling and print results for verification."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
        tools=[CodeInterpreterTool()] if CODE_TOOL_OK else [],
    )

    coding_task = Task(
        description=(
            "Write and run a Python script that calculates the first 10 "
            "Fibonacci numbers and their sum. Print the results clearly."
        ),
        expected_output=(
            "The first 10 Fibonacci numbers listed individually, "
            "followed by their sum."
        ),
        agent=coding_agent,
    )

    coding_crew = Crew(
        agents=[coding_agent],
        tasks=[coding_task],
        process=Process.sequential,
        verbose=False,
    )

    print("Running coding agent demo...")
    print("=" * 60)
    result = coding_crew.kickoff()
    print("=" * 60)
    print("\nAgent output:")
    print(str(result)[:500])
else:
    print("[SKIP] CrewAI or Ollama not available")

## 7. Manual Code Execution Fallback

When `CodeInterpreterTool` is not available, you can implement code
execution manually. This pattern is useful for understanding what the
tool does under the hood and for custom sandboxing requirements.

In [ ]:
# ---- Manual code execution demo -------------------------------------------

print("=== Manual Code Execution Demo ===\n")

# The agent "writes" this code (in reality, the LLM generates it).
agent_code = """
# Fibonacci calculation -- written by the coding agent
fib = [0, 1]
for i in range(2, 10):
    fib.append(fib[i-1] + fib[i-2])

total = sum(fib)
print("Fibonacci numbers:", fib)
print("Sum:", total)
"""

print("Agent-generated code:")
print("-" * 40)
print(agent_code)
print("-" * 40)

# Execute the code in a subprocess (safe execution pattern).
import subprocess
import tempfile

with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
    f.write(agent_code)
    tmp_path = f.name

try:
    result = subprocess.run(
        [sys.executable, tmp_path],
        capture_output=True, text=True, timeout=10
    )
    print("Execution result:")
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
    print(f"Return code: {result.returncode}")
finally:
    os.unlink(tmp_path)

## 8. Code Validation and Retry Pattern

Coding agents should validate their code before executing it. A common
pattern is: write code -> execute -> check output -> fix if needed -> rerun.

In [ ]:
# ---- Code validation pattern with retry -----------------------------------

print("=== Code Validation and Retry Pattern ===\n")

def validate_and_run(code: str, max_retries: int = 3) -> dict:
    """Execute code with validation and automatic retry on failure.

    Args:
        code: Python source code to execute.
        max_retries: Maximum number of retry attempts.

    Returns:
        Dictionary with final output and retry count.
    """
    for attempt in range(max_retries):
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(code)
            tmp_path = f.name

        try:
            result = subprocess.run(
                [sys.executable, tmp_path],
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                return {
                    "output": result.stdout,
                    "success": True,
                    "attempts": attempt + 1,
                }
            # On failure, the agent would fix the code (simulated here).
            print(f"  Attempt {attempt + 1} failed: {result.stderr[:100]}")
            code = code  # In reality, the LLM fixes the code.
        finally:
            os.unlink(tmp_path)

    return {"output": "", "success": False, "attempts": max_retries}

# Test with valid code.
valid_code = "print(sum(range(1, 101)))"
print(f"Running valid code: {valid_code}")
result = validate_and_run(valid_code)
print(f"  Success: {result['success']}, Attempts: {result['attempts']}")
print(f"  Output: {result['output'].strip()}")
print()

# Test with invalid code (will fail).
invalid_code = "print(undefined_variable)"
print(f"Running invalid code: {invalid_code}")
result = validate_and_run(invalid_code, max_retries=1)
print(f"  Success: {result['success']}, Attempts: {result['attempts']}")

## 9. CLI Commands Reference

Complete reference for CrewAI CLI commands relevant to production:

| Command | Purpose | Example |
|---------|---------|---------|
| `crewai create crew` | Scaffold a new project | `crewai create crew my_app` |
| `crewai run` | Execute a crew | `crewai run --inputs '{"k":"v"}'` |
| `crewai test` | Quick validation | `crewai test --crew MyCrew` |
| `crewai train` | Train crew prompts | `crewai train --n 5` |
| `crewai flow` | Run a Flow | `crewai flow --inputs '{}'` |

In [ ]:
# ---- CLI commands reference summary ---------------------------------------

print("=== CrewAI CLI Commands Reference ===\n")
commands = [
    ("crewai create crew NAME", "Scaffold a new crew project with standard structure"),
    ("crewai run", "Execute the crew defined in src/main.py"),
    ("crewai run --inputs '{...}'", "Execute with custom JSON inputs"),
    ("crewai test", "Run crew once for quick validation"),
    ("crewai test --crew NAME", "Test a specific crew by name"),
    ("crewai train --n NUM", "Train crew prompts over NUM iterations"),
    ("crewai flow", "Execute a Flow defined in the project"),
]
for cmd, desc in commands:
    print(f"  {cmd:40s} {desc}")

## Summary

This notebook covered CrewAI's coding agent and CLI capabilities:

1. **`CodeInterpreterTool`** -- sandboxed Python execution for coding agents.
2. **`AGENTS.md`** -- project configuration file for coding agents.
3. **`crewai create crew`** -- CLI scaffolding for new projects.
4. **`crewai run`** -- CLI execution of crews with inputs.
5. **Safe code execution** -- subprocess-based execution with timeout and output capture.
6. **Code validation** -- retry patterns for handling execution failures.

## Next steps

- Create a full crew project with `crewai create crew` and explore the structure.
- Combine `CodeInterpreterTool` with file tools for data processing agents.
- Use `AGENTS.md` to customize coding agent behavior for your project.